# **TrumorGPT: Dual-KB GraphRAG + SVM-SMOTE + TomekLinks**
### Imbalance Ablation Study on PUBHEALTH

**Experimental design.** We evaluate the same pipeline under two sampling regimes drawn from the PUBHEALTH dataset:

1. **Imbalanced (2:1)** — reflects natural class skew. Test set preserves the imbalance.
2. **Balanced (1:1)** — controls for the imbalance variable. Test set is balanced.

SVM-SMOTE + TomekLinks is applied to training data in both cases. This isolates the effect of class imbalance at evaluation time, holding pipeline and dataset domain constant.


In [ ]:
# CELL 1 — INSTALL (run once, then restart kernel)
!pip install -q sentence-transformers datasets matplotlib seaborn scikit-learn \
             pandas numpy xgboost lightgbm imbalanced-learn transformers \
             torch tqdm spacy networkx
!python -m spacy download en_core_web_sm
print("Install complete. Restart the kernel before continuing.")


In [ ]:
# CELL 2 — IMPORTS (single source of truth for libraries)
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, torch, spacy, networkx as nx, re, time, os
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix,
                             roc_curve, roc_auc_score, precision_recall_curve,
                             f1_score, recall_score)
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, StratifiedKFold
from imblearn.over_sampling import SVMSMOTE
from imblearn.under_sampling import TomekLinks
from transformers import pipeline
from tqdm import tqdm

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_colwidth", None)

# Single global spaCy load (reused across cells)
nlp = spacy.load("en_core_web_sm")

styles = [
    dict(selector="th", props=[("text-align","center"),("background-color","#f4f4f4")]),
    dict(selector="td", props=[("text-align","left")])
]

print("=" * 80)
print("[SYSTEM STARTED] TrumorGPT: Dual-KB GraphRAG + SVM-SMOTE + TomekLinks")
print("=" * 80)


In [ ]:
# CELL 3 — PUBHEALTH IMBALANCED (N=900: 600 True, 300 False)
print("\n[STEP 1] Downloading and sampling IMBALANCED PUBHEALTH data...")
try:
    dataset = load_dataset(
        "parquet",
        data_files="hf://datasets/health_fact@refs/convert/parquet/default/train/*.parquet",
        split="train"
    )
    df_raw = dataset.to_pandas()
except Exception:
    df_raw = pd.read_parquet(
        "https://huggingface.co/datasets/health_fact/resolve/"
        "refs%2Fconvert%2Fparquet/default/train/0000.parquet"
    )

df_raw = df_raw.rename(
    columns={"claim" if "claim" in df_raw.columns else "main_text": "claim_text"}
)

# Show what labels actually exist before mapping (transparency for reviewers)
print("  Raw label distribution:")
print(df_raw["label"].value_counts().to_string())

def map_label_pubhealth(x):
    val = str(x).lower().strip()
    if val in ["1", "2", "true", "mostly true"]: return 1
    elif val in ["0", "false", "pants on fire"]: return 0
    return None  # 'mixture', 'unproven', 3, etc. are dropped

df_raw["label"] = df_raw["label"].apply(map_label_pubhealth)
n_dropped = df_raw["label"].isna().sum()
df_raw = df_raw.dropna(subset=["claim_text", "label"])
print(f"  Dropped {n_dropped} rows with unmapped labels (mixture/unproven/etc.)")
print(f"  Mapped distribution: True={int((df_raw['label']==1).sum())}, "
      f"False={int((df_raw['label']==0).sum())}")

df_true  = df_raw[df_raw["label"] == 1].sample(600, random_state=42)
df_false = df_raw[df_raw["label"] == 0].sample(300, random_state=42)
df_claims = (pd.concat([df_true, df_false])
               .sample(frac=1, random_state=42)
               .reset_index(drop=True))

df_claims["claim_id"]   = [f"C{i+1:03d}" for i in range(len(df_claims))]
df_claims["claim_text"] = df_claims["claim_text"].astype(str).str.slice(0, 512)
df_claims["word_count"] = df_claims["claim_text"].str.split().str.len()

print(f"\n  Dataset      : PUBHEALTH (Imbalanced)")
print(f"  Total        : N = {len(df_claims)}")
print(f"  True claims  : {(df_claims['label']==1).sum()}")
print(f"  False claims : {(df_claims['label']==0).sum()}")
print(f"  Imbalance ratio: {(df_claims['label']==1).sum()/(df_claims['label']==0).sum():.1f}:1")

df_display = df_claims[['claim_id','claim_text','label']].head(10).copy()
df_display['label'] = df_display['label'].map({1:'True', 0:'False'})
display(
    df_display.style
    .set_caption(f"TABLE 1 — PUBHEALTH Imbalanced Sample (showing first 10 of {len(df_claims)})")
    .set_table_styles(styles).hide(axis="index")
)

df_train, df_test = train_test_split(
    df_claims, test_size=0.2, random_state=42, stratify=df_claims['label']
)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)
print(f"  Train: {len(df_train)} ({(df_train['label']==1).sum()}T / {(df_train['label']==0).sum()}F) | "
      f"Test: {len(df_test)} ({(df_test['label']==1).sum()}T / {(df_test['label']==0).sum()}F)")


In [ ]:
# CELL 4 — PUBHEALTH BALANCED (Imbalance Ablation)
# This is NOT a different dataset. Same source, balanced sampling strategy.
# The point: isolate the effect of class imbalance, not domain difficulty.
print("\n[STEP 2] Creating BALANCED benchmark from PUBHEALTH (same source)...")

n_balanced = min(
    int((df_raw["label"] == 1).sum()),
    int((df_raw["label"] == 0).sum()),
    300
)

bal_true  = df_raw[df_raw["label"] == 1].sample(n_balanced, random_state=99)
bal_false = df_raw[df_raw["label"] == 0].sample(n_balanced, random_state=99)
df_balanced_all = (pd.concat([bal_true, bal_false])
                     .sample(frac=1, random_state=99)
                     .reset_index(drop=True))
df_balanced_all["claim_text"] = df_balanced_all["claim_text"].astype(str).str.slice(0, 512)
df_balanced_all["word_count"] = df_balanced_all["claim_text"].str.split().str.len()
df_balanced_all["claim_id"]   = [f"B{i+1:03d}" for i in range(len(df_balanced_all))]

df_bal_train, df_bal_test = train_test_split(
    df_balanced_all, test_size=0.2, random_state=99, stratify=df_balanced_all["label"]
)
df_bal_train = df_bal_train.reset_index(drop=True)
df_bal_test  = df_bal_test.reset_index(drop=True)

print(f"  Dataset      : PUBHEALTH Balanced (same source, 1:1 sampling)")
print(f"  Train        : {len(df_bal_train)} ({(df_bal_train['label']==1).sum()} True | {(df_bal_train['label']==0).sum()} False)")
print(f"  Test         : {len(df_bal_test)}  ({(df_bal_test['label']==1).sum()} True  | {(df_bal_test['label']==0).sum()} False)")
print(f"  Balance ratio: 1:1")
print()
print("  [DESIGN NOTE] Same source, different sampling strategy.")
print("  Imbalanced (2:1) vs Balanced (1:1) — isolates CLASS IMBALANCE effect,")
print("  holding domain and pipeline constant.")


In [ ]:
# CELL 5 — SHKG (Semantic Health Knowledge Graph) — built from TRAINING DATA ONLY
# FIX: Previously used full df_claims (train + test). That leaks test info into PageRank.
# Now built per-split from training data only.
print("\n[STEP 3] Building SHKG (Entity Co-occurrence) — TRAIN ONLY...")

def build_shkg(claim_texts):
    """Build co-occurrence graph + PageRank from a list of claim texts."""
    G = nx.Graph()
    for claim in claim_texts:
        doc = nlp(claim)
        # Note: en_core_web_sm does NOT have a DISEASE entity type.
        # For real biomedical entities, scispaCy (en_ner_bc5cdr_md) would be needed.
        ents = list(set(ent.text.lower() for ent in doc.ents
                        if ent.label_ in ["ORG", "GPE", "PERSON", "PRODUCT"]))
        for i in range(len(ents)):
            for j in range(i + 1, len(ents)):
                if G.has_edge(ents[i], ents[j]):
                    G[ents[i]][ents[j]]['weight'] += 1
                else:
                    G.add_edge(ents[i], ents[j], weight=1)
    pr = nx.pagerank(G, weight='weight') if len(G.nodes) > 0 else {}
    return G, pr

# Imbalanced split SHKG (training only)
G_ent, pagerank_scores = build_shkg(df_train['claim_text'])
print(f"  [Imbalanced] SHKG: {len(G_ent.nodes())} nodes, {len(G_ent.edges())} edges")

# Balanced split SHKG (training only) — separate to avoid cross-split leakage
G_ent_bal, pagerank_scores_bal = build_shkg(df_bal_train['claim_text'])
print(f"  [Balanced]   SHKG: {len(G_ent_bal.nodes())} nodes, {len(G_ent_bal.edges())} edges")

# Visualize the imbalanced-split graph
G_vis = G_ent.copy()
G_vis.remove_nodes_from(list(nx.isolates(G_vis)))
node_sizes = [pagerank_scores.get(n, 0) * 90000 for n in G_vis.nodes()]
pos = nx.spring_layout(G_vis, k=0.25, iterations=100, seed=42)

plt.figure(figsize=(15, 11))
nx.draw(G_vis, pos, with_labels=True, node_color='lightsteelblue',
        node_size=node_sizes, font_size=7, font_weight='bold',
        edge_color='silver', width=1.5, alpha=0.9,
        linewidths=1.0, edgecolors='midnightblue')
plt.title(f"Semantic Health Knowledge Graph (Imbalanced split, train-only)\n"
          f"{len(G_vis.nodes())} connected entities from {len(df_train)} training claims",
          fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()


In [ ]:
# CELL 6 — DUAL-KB HYBRID FEATURE EXTRACTION (12-Dimensional)
# FIXES:
#   - PageRank passed as parameter (not global) — no cross-split leakage
#   - Self-match exclusion uses INDEX, not similarity threshold (more reliable)
#   - wc_norm clipped to [0, 1] for out-of-distribution queries
print("\n[STEP 4] Loading Models for Hybrid Feature Extraction...")

bert_model = SentenceTransformer("all-MiniLM-L6-v2")
device     = 0 if torch.cuda.is_available() else -1
# Using base instead of large: ~3x faster, comparable NLI accuracy
nli_model  = pipeline("zero-shot-classification",
                      model="cross-encoder/nli-deberta-v3-base", device=device)

HEDGE_WORDS  = {'might', 'could', 'may', 'possibly', 'allegedly', 'claims',
                'rumored', 'apparently', 'supposedly', 'unconfirmed'}
SENSATIONAL  = {'secret', 'shocking', 'miracle', 'cure', 'deadly', 'banned',
                'hidden', 'exposed', 'bombshell', 'urgent', 'exclusive', 'breaking'}

FEAT_NAMES = ["cos_true", "sup_true", "con_true",
              "cos_false", "sup_false", "con_false",
              "wc_norm", "graph_score",
              "hedge_score", "sens_score", "num_score", "question_flag"]


def extract_hybrid_features(claim, claim_id, word_count,
                            kb_t_texts, kb_t_vecs,
                            kb_f_texts, kb_f_vecs,
                            wc_maximum, pagerank_dict,
                            train_idx_in_true=None, train_idx_in_false=None,
                            log_list=None):
    """
    Extract 12-D feature vector for one claim.

    train_idx_in_true / train_idx_in_false: if provided, exclude that exact
    KB position from retrieval (prevents self-match leakage during training).
    """
    query_vec = normalize(bert_model.encode([claim], show_progress_bar=False), norm="l2")

    # True KB retrieval
    sims_true = (query_vec @ kb_t_vecs.T)[0]
    if train_idx_in_true is not None and 0 <= train_idx_in_true < len(sims_true):
        sims_true[train_idx_in_true] = -1.0
    best_true_idx = int(np.argmax(sims_true))
    cos_true      = float(sims_true[best_true_idx])
    context_true  = kb_t_texts[best_true_idx]

    # False KB retrieval
    sims_false = (query_vec @ kb_f_vecs.T)[0]
    if train_idx_in_false is not None and 0 <= train_idx_in_false < len(sims_false):
        sims_false[train_idx_in_false] = -1.0
    best_false_idx = int(np.argmax(sims_false))
    cos_false      = float(sims_false[best_false_idx])
    context_false  = kb_f_texts[best_false_idx]

    # NLI features
    out_true  = nli_model(f"Medical Fact: {context_true}\nClaim: {claim}",
                          candidate_labels=["Supports", "Contradicts", "Neutral"])
    sc_true   = dict(zip(out_true['labels'], out_true['scores']))
    sup_true, con_true = sc_true.get("Supports", 0.0), sc_true.get("Contradicts", 0.0)

    out_false = nli_model(f"Fake Health Claim: {context_false}\nClaim: {claim}",
                          candidate_labels=["Supports", "Contradicts", "Neutral"])
    sc_false  = dict(zip(out_false['labels'], out_false['scores']))
    sup_false, con_false = sc_false.get("Supports", 0.0), sc_false.get("Contradicts", 0.0)

    # Graph score (uses split-specific PageRank passed as parameter)
    doc         = nlp(claim)
    claim_nodes = [ent.text.lower() for ent in doc.ents]
    graph_score = float(np.mean([pagerank_dict.get(n, 0) for n in claim_nodes])) if claim_nodes else 0.0

    # Word count, clipped (handles out-of-distribution live queries)
    wc_norm = min(word_count / wc_maximum, 1.0)

    # Linguistic features
    tokens      = claim.split()
    n_tok       = len(tokens) + 1
    hedge_score = sum(1 for w in tokens if w.lower() in HEDGE_WORDS) / n_tok
    sens_score  = sum(1 for w in tokens if w.lower() in SENSATIONAL) / n_tok
    num_score   = len(re.findall(r'\d+\.?\d*%?', claim)) / n_tok
    q_flag      = 1.0 if '?' in claim else 0.0

    if log_list is not None:
        log_list.append({
            "Claim_ID":        claim_id,
            "User_Claim":      claim[:55] + "...",
            "cos_true":        round(cos_true, 4),
            "sup_true":        round(sup_true, 4),
            "con_true":        round(con_true, 4),
            "cos_false":       round(cos_false, 4),
            "graph_score":     round(graph_score, 6),
            "hedge_score":     round(hedge_score, 4),
            "Best_True_Fact":  context_true[:80],
            "Best_False_Fact": context_false[:80],
        })

    return [cos_true, sup_true, con_true, cos_false, sup_false, con_false,
            wc_norm, graph_score, hedge_score, sens_score, num_score, q_flag]


def extract_features_for_split(df_train_split, df_test_split,
                                pagerank_dict, log_list=None):
    """
    Build per-split KB vectors and extract train+test features.
    Eliminates duplicate code between cells 6 and 7.
    """
    kb_t_texts = df_train_split[df_train_split['label'] == 1]["claim_text"].tolist()
    kb_t_vecs  = normalize(bert_model.encode(kb_t_texts, show_progress_bar=False), norm="l2")
    kb_f_texts = df_train_split[df_train_split['label'] == 0]["claim_text"].tolist()
    kb_f_vecs  = normalize(bert_model.encode(kb_f_texts, show_progress_bar=False), norm="l2")
    wc_max     = df_train_split["word_count"].max()  # Use TRAIN max (not full data)

    # Build index maps so each training claim knows its position in the KB
    true_id_to_idx  = {cid: i for i, cid in enumerate(
        df_train_split[df_train_split['label'] == 1]["claim_id"].tolist())}
    false_id_to_idx = {cid: i for i, cid in enumerate(
        df_train_split[df_train_split['label'] == 0]["claim_id"].tolist())}

    print(f"    Train KB: {len(kb_t_texts)} True, {len(kb_f_texts)} False vectors")
    print("    Extracting TRAIN features...")
    X_tr = np.array([
        extract_hybrid_features(
            row["claim_text"], row["claim_id"], row["word_count"],
            kb_t_texts, kb_t_vecs, kb_f_texts, kb_f_vecs, wc_max, pagerank_dict,
            train_idx_in_true=true_id_to_idx.get(row["claim_id"]) if row["label"] == 1 else None,
            train_idx_in_false=false_id_to_idx.get(row["claim_id"]) if row["label"] == 0 else None,
            log_list=None
        )
        for _, row in tqdm(df_train_split.iterrows(), total=len(df_train_split))
    ])
    y_tr = df_train_split["label"].values

    print("    Extracting TEST features...")
    X_te = np.array([
        extract_hybrid_features(
            row["claim_text"], row["claim_id"], row["word_count"],
            kb_t_texts, kb_t_vecs, kb_f_texts, kb_f_vecs, wc_max, pagerank_dict,
            log_list=log_list
        )
        for _, row in tqdm(df_test_split.iterrows(), total=len(df_test_split))
    ])
    y_te = df_test_split["label"].values

    kb_artifacts = {
        "kb_t_texts": kb_t_texts, "kb_t_vecs": kb_t_vecs,
        "kb_f_texts": kb_f_texts, "kb_f_vecs": kb_f_vecs,
        "wc_max": wc_max,
    }
    return X_tr, y_tr, X_te, y_te, kb_artifacts


reasoning_logs = []
print("\n  >>> Imbalanced split <<<")
X_train, y_train, X_test, y_test, kb_imb = extract_features_for_split(
    df_train, df_test, pagerank_scores, log_list=reasoning_logs)

print(f"\n  Imbalanced — Train: {X_train.shape} | Test: {X_test.shape}")

# Optional: cache to disk so re-running downstream cells doesn't redo extraction
np.savez("trumorgpt_features_imbalanced.npz",
         X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test)
print("  Cached features → trumorgpt_features_imbalanced.npz")

df_table2 = pd.DataFrame(reasoning_logs).head(15)
display(
    df_table2.style
    .set_caption("TABLE 2 — Dual-KB RAG Reasoning Logs (first 15 test claims)")
    .background_gradient(subset=['con_true'],   cmap='Reds')
    .background_gradient(subset=['sup_true'],   cmap='Greens')
    .background_gradient(subset=['graph_score'], cmap='Purples')
    .set_table_styles(styles).hide(axis="index")
)


In [ ]:
# CELL 7 — BALANCED SPLIT FEATURE EXTRACTION (uses Cell 6 helper)
print("\n[STEP 5] Extracting features for BALANCED split...")
print("  >>> Balanced split <<<")
X_bal_train, y_bal_train, X_bal_test, y_bal_test, kb_bal = extract_features_for_split(
    df_bal_train, df_bal_test, pagerank_scores_bal, log_list=None)

print(f"\n  Balanced — Train: {X_bal_train.shape} | Test: {X_bal_test.shape}")

np.savez("trumorgpt_features_balanced.npz",
         X_train=X_bal_train, y_train=y_bal_train,
         X_test=X_bal_test, y_test=y_bal_test)
print("  Cached features → trumorgpt_features_balanced.npz")


In [ ]:
# CELL 8 — VISUALIZATIONS (Entity Analysis & KB Diff)
print("\n[STEP 6] Generating Entity Analysis & Logic Distribution Plot...")

# Entity counts from TRAIN only (not full data — consistent with leak fix)
all_train_text = " ".join(df_train["claim_text"].tolist())
ent_doc        = nlp(all_train_text)
ents           = [e.text.lower() for e in ent_doc.ents
                  if e.label_ in ["ORG", "GPE", "PERSON", "PRODUCT"]]
top_entities   = pd.Series(ents).value_counts().head(10)

df_plot = pd.DataFrame(X_test, columns=FEAT_NAMES)
df_plot['label']        = pd.Series(y_test).map({1: 'True', 0: 'False'}).values
df_plot['Logic_Margin'] = df_plot['sup_true'] - df_plot['con_true']
df_plot['KB_Diff']      = df_plot['cos_true'] - df_plot['cos_false']
df_plot['word_count']   = df_test['word_count'].values

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(x=top_entities.values, y=top_entities.index, ax=axes[0], palette="viridis")
axes[0].set_title(f"Top 10 Extracted Entities (Imbalanced train, N={len(df_train)})",
                  fontweight='bold')
axes[0].set_xlabel("Frequency")

sns.violinplot(data=df_plot, x="label", y="KB_Diff",
               palette={"True": "mediumseagreen", "False": "indianred"},
               ax=axes[1], inner="quart")
axes[1].set_title("KB Proximity Difference — True vs False Claims\n"
                  "(KB_Diff = cos_true − cos_false)", fontweight='bold')
axes[1].axhline(0, color='navy', linestyle='--', linewidth=1)
plt.tight_layout(); plt.show()


In [ ]:
# CELL 9 — WORD COUNT & LOGIC MARGIN VISUALIZATIONS
print("\n[STEP 7] Word Count Analysis & Relevance Score Distributions...")

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
avg_wc = df_plot.groupby("label")["word_count"].mean()
sns.barplot(x=avg_wc.index, y=avg_wc.values,
            palette={"True": "mediumseagreen", "False": "indianred"},
            ax=axes[0], edgecolor='black')
axes[0].set_title(f"Average Word Count per Claim (Test set, N={len(df_test)})",
                  fontweight='bold')
axes[0].set_ylabel("Average Number of Words")
for i, v in enumerate(avg_wc.values):
    axes[0].text(i, v + 0.5, f"{v:.1f}", ha='center', fontweight='bold', fontsize=12)

sns.scatterplot(data=df_plot, x="word_count", y="Logic_Margin", hue="label",
                palette={"True": "mediumseagreen", "False": "indianred"},
                alpha=0.6, ax=axes[1])
sns.regplot(data=df_plot, x="word_count", y="Logic_Margin", scatter=False,
            color="black", line_kws={"linestyle": "--"}, ax=axes[1])
axes[1].set_title("Claim Length vs Logical Reasoning Score", fontweight='bold')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.boxplot(data=df_plot, x="label", y="KB_Diff", order=['True', 'False'],
            palette={"True": "mediumseagreen", "False": "indianred"},
            ax=axes[0], linewidth=1.5)
axes[0].set_title("KB Proximity Difference Statistical Summary", fontweight='bold')
axes[0].axhline(0, color='navy', linestyle='--', linewidth=1)

sns.scatterplot(data=df_plot, x="KB_Diff", y="cos_true", hue="label",
                palette={"True": "mediumseagreen", "False": "indianred"},
                ax=axes[1], alpha=0.7, s=60)
axes[1].set_title("KB_Diff vs True Cosine — Discriminative Space", fontweight='bold')
axes[1].axvline(0, color='navy', linestyle='--', linewidth=1)
plt.tight_layout(); plt.show()


In [ ]:
# CELL 10 — PIPELINE: Our Method + Paper Method (4 combinations)
#
# Combination 1: Paper Method (Jaccard GraphRAG, no SMOTE)  — Balanced
# Combination 2: Paper Method + SMOTE+Tomek                 — Imbalanced
# Combination 3: Our Method (Dual-KB + Bagging-AdaBoost)    — Balanced
# Combination 4: Our Method (Dual-KB + Bagging-AdaBoost)    — Imbalanced
#
# Dataset is fixed (PUBHEALTH), method and class balance are changing.
# This design isolates both the method contribution and the class imbalance effect.

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: Jaccard triple overlap (BERT-triple equivalent of article Eq. 5)
# Article: S(Gx, Gi) = |Tx ∩ Ti| / |Tx ∪ Ti| (Jaccard on consecutive triples)
# Here: we create triples with spaCy NEs (head, relation, tail),
# then we match with the most similar KB input using Jaccard similarity ->.
# ─────────────────────────────────────────────────────────────────────────────
def extract_triples_from_text(text):
    doc = nlp(text)
    triples = set()
    ents = [e.text.lower() for e in doc.ents]
    for chunk in doc.noun_chunks:
        head = chunk.root.head
        if head.dep_ in ('nsubj', 'nsubjpass', 'dobj', 'pobj', 'attr'):
            subj = chunk.text.lower()
            rel  = head.lemma_.lower()
            obj  = head.head.text.lower() if head.head != head else 'unknown'
            triples.add((subj, rel, obj))
    # Entity co-occurrence triples (The article is like the example in Fig. 2.)
    for i in range(len(ents)):
        for j in range(i+1, len(ents)):
            triples.add((ents[i], 'co-occurs', ents[j]))
    return triples


def jaccard_similarity(triples_x, triples_i):
    if not triples_x and not triples_i:
        return 0.0
    intersection = len(triples_x & triples_i)
    union        = len(triples_x | triples_i)
    return intersection / union if union > 0 else 0.0


def build_kb_triples(df_split):
    kb_true_triples  = [extract_triples_from_text(t)
                        for t in df_split[df_split['label']==1]['claim_text'].tolist()]
    kb_false_triples = [extract_triples_from_text(t)
                        for t in df_split[df_split['label']==0]['claim_text'].tolist()]
    return kb_true_triples, kb_false_triples


def predict_jaccard(claim_text, kb_true_triples, kb_false_triples):
    query_triples = extract_triples_from_text(claim_text)
    best_true  = max((jaccard_similarity(query_triples, t) for t in kb_true_triples),
                     default=0.0)
    best_false = max((jaccard_similarity(query_triples, t) for t in kb_false_triples),
                     default=0.0)
    # Makale: S(Gx, Gi) <= theta => True (falling below the low similarity threshold)
    # Biz: best_true > best_false => True (higher similarity True KB)
    score = best_true - best_false   # positive => True direction
    return score, best_true, best_false


# ─────────────────────────────────────────────────────────────────────────────
# ARTICLE METHOD (Jaccard GraphRAG)
# NO SMOTE/Tomek — the article does not use these techniques.
# We will only add SMOTE+Tomek to the imbalanced version (combination 2).
# ─────────────────────────────────────────────────────────────────────────────
def paper_method_evaluate(df_train_split, df_test_split,
                           dataset_name, apply_smote=False, verbose=True):
    if verbose:
        print(f"\n{'='*65}")
        smote_tag = ' + SMOTE+Tomek' if apply_smote else ' (no resampling)'
        print(f"PAPER METHOD{smote_tag} — {dataset_name}")
        print(f"{'='*65}")
        print(f"  Train: {(df_train_split['label']==1).sum()} True | "
              f"{(df_train_split['label']==0).sum()} False")
        print(f"  Test : {(df_test_split['label']==1).sum()} True | "
              f"{(df_test_split['label']==0).sum()} False")
        print(f"  Scoring: Jaccard triple-overlap similarity (paper Eq.5 analogue)")

    # Create KB from train.
    print("  Building KB triples from training data...")
    kb_true_triples, kb_false_triples = build_kb_triples(df_train_split)

    # Calculate Jaccard scores on the test set.
    print("  Computing Jaccard scores on test set...")
    scores, y_true = [], []
    for _, row in df_test_split.iterrows():
        sc, _, _ = predict_jaccard(row['claim_text'], kb_true_triples, kb_false_triples)
        scores.append(sc)
        y_true.append(int(row['label']))

    scores  = np.array(scores)
    y_true  = np.array(y_true)

    # Threshold tuning: on the train set
    tr_scores, tr_labels = [], []
    for _, row in df_train_split.iterrows():
        sc, _, _ = predict_jaccard(row['claim_text'], kb_true_triples, kb_false_triples)
        tr_scores.append(sc)
        tr_labels.append(int(row['label']))
    tr_scores = np.array(tr_scores)
    tr_labels = np.array(tr_labels)

    # Apply SMOTE+Tomek (combination 2 only: imbalanced + paper method)
    if apply_smote:
        print("  Applying SVM-SMOTE + TomekLinks on Jaccard score feature...")
        sc_2d = tr_scores.reshape(-1, 1)
        smote_obj = SVMSMOTE(sampling_strategy='auto', random_state=42,
                             k_neighbors=min(5, (tr_labels==0).sum()-1),
                             m_neighbors=10,
                             svm_estimator=SVC(kernel='rbf', gamma='scale',
                                               random_state=42))
        sc_res, lab_res = smote_obj.fit_resample(sc_2d, tr_labels)
        tomek_obj = TomekLinks(sampling_strategy='majority')
        sc_res, lab_res = tomek_obj.fit_resample(sc_res, lab_res)
        tr_scores_use = sc_res.ravel()
        tr_labels_use = lab_res
        print(f"  After SMOTE+Tomek: {(tr_labels_use==1).sum()} True | "
              f"{(tr_labels_use==0).sum()} False")
    else:
        tr_scores_use = tr_scores
        tr_labels_use = tr_labels

    # Find threshold from validation slice
    idx_fit, idx_val = train_test_split(
        range(len(tr_labels_use)), test_size=0.2,
        random_state=42, stratify=tr_labels_use)
    val_scores = tr_scores_use[list(idx_val)]
    val_labels = tr_labels_use[list(idx_val)]

    best_thr, best_f1 = 0.0, 0.0
    for thr in np.linspace(tr_scores_use.min(), tr_scores_use.max(), 300):
        preds  = (val_scores >= thr).astype(int)
        f1_val = f1_score(val_labels, preds, average='macro', zero_division=0)
        if f1_val > best_f1:
            best_f1, best_thr = f1_val, float(thr)

    y_pred  = (scores >= best_thr).astype(int)
    scale   = max(abs(scores).max(), 1e-6)
    y_proba = 1 / (1 + np.exp(-scores / scale * 4))  # sigmoid pseudo-proba

    metrics = {
        'dataset':         dataset_name,
        'accuracy':        accuracy_score(y_true, y_pred),
        'f1_macro':        f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted':     f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_minority': recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        'recall_majority': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'auc':             roc_auc_score(y_true, y_proba),
        'threshold':       best_thr,
        'y_pred':          y_pred,
        'y_proba':         y_proba,
        'y_true':          y_true,
    }

    if verbose:
        print(f"  Optimal threshold: {best_thr:.4f}")
        print(f"  Accuracy  : {metrics['accuracy']:.3f}")
        print(f"  F1 (macro): {metrics['f1_macro']:.3f}")
        print(f"  AUC-ROC   : {metrics['auc']:.3f}")
        print()
        print(classification_report(y_true, y_pred, zero_division=0,
                                    target_names=['False', 'True']))
    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# OUR METHOD (Dual-KB + SVM-SMOTE + Bagging-AdaBoost)
# ─────────────────────────────────────────────────────────────────────────────
def train_and_evaluate(X_tr, y_tr, X_te, y_te, dataset_name,
                        random_state=42, verbose=True):
    if verbose:
        print(f"\n{'='*65}")
        print(f"OUR METHOD (Dual-KB + SVM-SMOTE + Bagging-AdaBoost) — {dataset_name}")
        print(f"{'='*65}")
        print(f"  Train: {(y_tr==1).sum()} True | {(y_tr==0).sum()} False")
        print(f"  Test : {(y_te==1).sum()} True | {(y_te==0).sum()} False")

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)

    X_tr_fit, X_val, y_tr_fit, y_val = train_test_split(
        X_tr_s, y_tr, test_size=0.2, random_state=random_state, stratify=y_tr)

    smote = SVMSMOTE(sampling_strategy='auto', random_state=random_state,
                     k_neighbors=5, m_neighbors=10,
                     svm_estimator=SVC(kernel='rbf', gamma='scale',
                                       random_state=random_state))
    X_res, y_res = smote.fit_resample(X_tr_fit, y_tr_fit)
    tomek = TomekLinks(sampling_strategy='majority')
    X_res, y_res = tomek.fit_resample(X_res, y_res)

    if verbose:
        print(f"  After SMOTE+Tomek: {(y_res==1).sum()} True | {(y_res==0).sum()} False")

    base_tree = DecisionTreeClassifier(max_depth=1, max_features='sqrt',
                                       random_state=random_state)
    base_ada  = AdaBoostClassifier(estimator=base_tree, n_estimators=50,
                                   random_state=random_state)
    model     = BaggingClassifier(estimator=base_ada, n_estimators=10,
                                  max_samples=0.8, random_state=random_state,
                                  n_jobs=-1)

    cv_scores = cross_val_score(model, X_res, y_res, cv=5, scoring='f1_macro')
    if verbose:
        print(f"  5-Fold CV macro-F1: {cv_scores.mean():.3f} (+/- {cv_scores.std()*2:.3f})")

    model.fit(X_res, y_res)

    val_proba = model.predict_proba(X_val)[:, 1]
    prec_v, rec_v, thr_v = precision_recall_curve(y_val, val_proba)
    f1_v    = 2 * (prec_v * rec_v) / (prec_v + rec_v + 1e-9)
    best_idx = int(np.argmax(f1_v[:-1])) if len(thr_v) > 0 else 0
    best_thr = float(thr_v[best_idx]) if len(thr_v) > 0 else 0.5
    if verbose:
        print(f"  Optimal threshold: {best_thr:.4f}")

    test_proba = model.predict_proba(X_te_s)[:, 1]
    y_pred     = (test_proba >= best_thr).astype(int)

    metrics = {
        'dataset':         dataset_name,
        'accuracy':        accuracy_score(y_te, y_pred),
        'f1_macro':        f1_score(y_te, y_pred, average='macro', zero_division=0),
        'f1_weighted':     f1_score(y_te, y_pred, average='weighted', zero_division=0),
        'recall_minority': recall_score(y_te, y_pred, pos_label=0, zero_division=0),
        'recall_majority': recall_score(y_te, y_pred, pos_label=1, zero_division=0),
        'auc':             roc_auc_score(y_te, test_proba),
        'threshold':       best_thr,
        'y_pred':          y_pred,
        'y_proba':         test_proba,
        'model':           model,
        'scaler':          sc,
    }

    if verbose:
        print(f"\n  RESULTS — {dataset_name}")
        print(f"  Accuracy        : {metrics['accuracy']:.3f}")
        print(f"  F1 (macro)      : {metrics['f1_macro']:.3f}")
        print(f"  F1 (weighted)   : {metrics['f1_weighted']:.3f}")
        print(f"  Recall (False)  : {metrics['recall_minority']:.3f}")
        print(f"  AUC-ROC         : {metrics['auc']:.3f}")
        print()
        print(classification_report(y_te, y_pred, zero_division=0,
                                    target_names=['False', 'True']))
    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# RUN 4 COMBINATIONS
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "#"*70)
print("COMBINATION 1: Paper Method (Jaccard GraphRAG, no SMOTE) — BALANCED")
print("#"*70)
results_paper_balanced = paper_method_evaluate(
    df_bal_train, df_bal_test,
    dataset_name='PUBHEALTH Balanced (1:1)',
    apply_smote=False)

print("\n" + "#"*70)
print("COMBINATION 2: Paper Method + SMOTE+Tomek — IMBALANCED")
print("#"*70)
results_paper_imbalanced = paper_method_evaluate(
    df_train, df_test,
    dataset_name='PUBHEALTH Imbalanced (2:1)',
    apply_smote=True)

print("\n" + "#"*70)
print("COMBINATION 3: Our Method (Dual-KB + SVM-SMOTE + Bagging) — BALANCED")
print("#"*70)
results_balanced = train_and_evaluate(
    X_bal_train, y_bal_train, X_bal_test, y_bal_test,
    'PUBHEALTH Balanced (1:1)')

print("\n" + "#"*70)
print("COMBINATION 4: Our Method (Dual-KB + SVM-SMOTE + Bagging) — IMBALANCED")
print("#"*70)
results_imbalanced = train_and_evaluate(
    X_train, y_train, X_test, y_test,
    'PUBHEALTH Imbalanced (2:1)')


In [ ]:
# CELL 11 - HYBRID RESAMPLING PIPELINE: Handling Class Imbalance with SVM-SMOTE & TomekLinks
print("\n[STEP 8] Applying SVM-SMOTE + TomekLinks — PUBHEALTH (Imbalanced)...")

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

print(f"  Before: {(y_train==1).sum()} True | {(y_train==0).sum()} False")

svm_smote = SVMSMOTE(sampling_strategy='auto', random_state=42, k_neighbors=5,
                     m_neighbors=10, svm_estimator=SVC(kernel='rbf', gamma='scale', random_state=42))
X_res, y_res = svm_smote.fit_resample(X_train_scaled, y_train)
print(f"  After SVM-SMOTE  : {(y_res==1).sum()} True | {(y_res==0).sum()} False")

tomek = TomekLinks(sampling_strategy='all')
X_res_clean, y_res_clean = tomek.fit_resample(X_res, y_res)
print(f"  After TomekLinks : {(y_res_clean==1).sum()} True | {(y_res_clean==0).sum()} False")

fig, axes = plt.subplots(1, 3, figsize=(18,5))
def plot_hybrid_space(ax, X, y, title):
    kb_diff = X[:,0] - X[:,3]
    margin  = X[:,1] - X[:,2]
    sns.scatterplot(x=kb_diff[y==1], y=margin[y==1], ax=ax,
                    color='mediumseagreen', label='True Fact (1)', alpha=0.6, s=50)
    sns.scatterplot(x=kb_diff[y==0], y=margin[y==0], ax=ax,
                    color='indianred', label='Fake News (0)', alpha=0.9, s=50, edgecolor='black')
    ax.axvline(0, color='navy', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('KB_Diff'); ax.set_ylabel('Logic Margin'); ax.legend()

plot_hybrid_space(axes[0], X_train_scaled, y_train, "1. Imbalanced")
plot_hybrid_space(axes[1], X_res,           y_res,   "2. After SVM-SMOTE")
plot_hybrid_space(axes[2], X_res_clean, y_res_clean, "3. After TomekLinks")
plt.tight_layout(); plt.savefig("fig1_hybrid_space.png", dpi=200); plt.show()

In [ ]:
# CELL 12 — SVM DECISION BOUNDARY (Before vs After Resampling)
# Visualization only — uses the imbalanced split.
print("\n[STEP 9] SVM Decision Boundary: Original vs Resampled...")

scaler_vis  = StandardScaler()
X_train_sc  = scaler_vis.fit_transform(X_train)

# Manual SMOTE for visualization (no val split — we want to see the FULL effect)
smote_vis = SVMSMOTE(sampling_strategy='auto', random_state=42, k_neighbors=5,
                     m_neighbors=10, svm_estimator=SVC(kernel='rbf', gamma='scale',
                                                       random_state=42))
X_res_vis, y_res_vis = smote_vis.fit_resample(X_train_sc, y_train)
tomek_vis = TomekLinks(sampling_strategy='majority')
X_res_vis, y_res_vis = tomek_vis.fit_resample(X_res_vis, y_res_vis)

print(f"  Imbalanced train: {(y_train==1).sum()}T / {(y_train==0).sum()}F")
print(f"  After SMOTE+Tomek: {(y_res_vis==1).sum()}T / {(y_res_vis==0).sum()}F")

def plot_boundary(ax, svc, X_vis, y, title):
    x_min, x_max = X_vis[:, 0].min() - 0.3, X_vis[:, 0].max() + 0.3
    y_min, y_max = X_vis[:, 1].min() - 0.3, X_vis[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                          np.linspace(y_min, y_max, 300))
    Z = svc.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    contour = ax.contourf(xx, yy, Z, alpha=0.18, levels=np.linspace(0, 1, 11), cmap='RdYlGn')
    plt.colorbar(contour, ax=ax, label='P(True Fact)')
    ax.contour(xx, yy, Z, levels=[0.5], colors='navy', linewidths=2.0, linestyles='--')
    ax.scatter(X_vis[y == 1, 0], X_vis[y == 1, 1], c='mediumseagreen',
               alpha=0.4, s=35, label=f'True (N={(y==1).sum()})', zorder=2)
    ax.scatter(X_vis[y == 0, 0], X_vis[y == 0, 1], c='indianred',
               alpha=0.8, s=50, edgecolors='black',
               label=f'False (N={(y==0).sum()})', zorder=3)
    sv_idx = svc.support_
    sv_min = [i for i in sv_idx if y[i] == 0]
    if sv_min:
        ax.scatter(X_vis[sv_min, 0], X_vis[sv_min, 1], c='gold', s=110,
                   edgecolors='black', linewidths=1.5, zorder=4, marker='*',
                   label=f'Support Vectors (N={len(sv_min)})')
    ax.axvline(0, color='gray', linestyle=':', linewidth=1)
    ax.set_xlabel('KB_Diff (cos_true − cos_false)', fontweight='bold')
    ax.set_ylabel('Logic Margin (sup_true − con_true)', fontweight='bold')
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.legend(loc='upper right', frameon=True, shadow=True, fontsize=8)


kb_orig = X_train_sc[:, 0] - X_train_sc[:, 3]
mg_orig = X_train_sc[:, 1] - X_train_sc[:, 2]
X_vis_orig = np.column_stack([kb_orig, mg_orig])
svc_orig   = SVC(kernel='rbf', gamma='scale', C=1.0, probability=True, random_state=42)
svc_orig.fit(X_vis_orig, y_train)

kb_res = X_res_vis[:, 0] - X_res_vis[:, 3]
mg_res = X_res_vis[:, 1] - X_res_vis[:, 2]
X_vis_res = np.column_stack([kb_res, mg_res])
svc_res   = SVC(kernel='rbf', gamma='scale', C=1.0, probability=True, random_state=42)
svc_res.fit(X_vis_res, y_res_vis)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
plot_boundary(axes[0], svc_orig, X_vis_orig, y_train,
              f"BEFORE Resampling\nTrue:{(y_train==1).sum()} | False:{(y_train==0).sum()} — Imbalanced")
plot_boundary(axes[1], svc_res, X_vis_res, y_res_vis,
              f"AFTER SVM-SMOTE + TomekLinks\nTrue:{(y_res_vis==1).sum()} | False:{(y_res_vis==0).sum()} — Balanced")
plt.suptitle("SVM Decision Boundary — Before vs After Resampling",
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("fig_svmsmote_boundary.png", dpi=200, bbox_inches='tight'); plt.show()


In [ ]:
# CELL 13 — REAL PAGERANK CONVERGENCE (was: synthetic exponential plot)
# FIX: Previous version drew a hand-coded exponential and labeled it as
# "PageRank convergence". This version measures actual L1 error per iteration
# on the SHKG graph for several damping factors.
print("\n[STEP 10] Real PageRank convergence on SHKG (per damping factor)...")

def pagerank_with_history(G, alpha=0.85, max_iter=100, tol=1e-12, weight='weight'):
    """Custom power-iteration PageRank that records L1 error per step."""
    if len(G) == 0:
        return {}, []
    nodes = list(G.nodes())
    n = len(nodes)
    idx = {node: i for i, node in enumerate(nodes)}

    # Build column-stochastic transition matrix
    M = np.zeros((n, n))
    for u in nodes:
        out_w = sum(G[u][v].get(weight, 1.0) for v in G[u])
        if out_w == 0:
            continue
        for v in G[u]:
            M[idx[v], idx[u]] = G[u][v].get(weight, 1.0) / out_w
    # For undirected graphs, ensure column-stochastic
    col_sums = M.sum(axis=0)
    M[:, col_sums == 0] = 1.0 / n  # dangling node fix

    x = np.ones(n) / n
    teleport = np.ones(n) / n
    history = []
    for _ in range(max_iter):
        x_new = alpha * (M @ x) + (1 - alpha) * teleport
        err = np.abs(x_new - x).sum()
        history.append(err)
        x = x_new
        if err < tol:
            break
    return dict(zip(nodes, x)), history


plt.figure(figsize=(10, 5))
for d in [0.60, 0.70, 0.85, 0.95]:
    _, history = pagerank_with_history(G_ent, alpha=d, max_iter=80, tol=1e-15)
    plt.plot(range(1, len(history) + 1), history, label=f"damping = {d}",
             linewidth=2, marker='o', markersize=3)

plt.yscale("log")
plt.title("Measured PageRank Convergence on SHKG\n(L1 error per iteration, real graph)",
          fontweight='bold')
plt.ylabel("L1 error  ||x⁽ᵗ⁺¹⁾ − x⁽ᵗ⁾||₁  (log scale)")
plt.xlabel("Iteration t")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("[Note] Lower damping (d=0.6) → faster convergence but less weight on link structure.")
print("       Higher damping (d=0.95) → slower convergence but stronger topology signal.")


In [ ]:
# ============================================================
# CELL 14 — REAL PAGERANK CONVERGENCE FOR TRUMORGPT (TST)
# Self-contained version (independent from Cell 24)
# ============================================================

print("\n[STEP 11] Real PageRank convergence on TST graph...")

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.preprocessing import normalize
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# ------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------

ETA = 0.7          # BERT-LDA fusion coefficient
N_TOPICS = 8       # number of LDA topics

# ------------------------------------------------------------
# SAMPLE CLAIMS
# ------------------------------------------------------------

sample_claims = (
    df_train['claim_text']
    .dropna()
    .sample(40, random_state=42)
    .tolist()
)

# ------------------------------------------------------------
# LDA MODEL (SELF-CONTAINED)
# ------------------------------------------------------------

vectorizer_tst = CountVectorizer(
    stop_words='english',
    max_features=2000
)

X_bow_tst = vectorizer_tst.fit_transform(sample_claims)

lda_tst = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42
)

lda_tst.fit(X_bow_tst)

# ------------------------------------------------------------
# HELPER FUNCTION
# ------------------------------------------------------------

def get_lda_vec(texts):
    bow = vectorizer_tst.transform(texts)
    return lda_tst.transform(bow)

# ------------------------------------------------------------
# BERT EMBEDDINGS
# ------------------------------------------------------------

bert_embs = normalize(
    bert_model.encode(
        sample_claims,
        show_progress_bar=False
    ),
    norm='l2'
)

# ------------------------------------------------------------
# LDA TOPIC EMBEDDINGS
# ------------------------------------------------------------

lda_vecs = normalize(
    get_lda_vec(sample_claims),
    norm='l2'
)

# ------------------------------------------------------------
# TST REPRESENTATION
# ------------------------------------------------------------

combined = np.concatenate(
    [
        ETA * bert_embs,
        (1 - ETA) * lda_vecs
    ],
    axis=1
)

# ------------------------------------------------------------
# SIMILARITY GRAPH
# ------------------------------------------------------------

sim = combined @ combined.T

np.fill_diagonal(sim, 0)

sim = np.clip(sim, 0, None)

G_tst = nx.from_numpy_array(sim)

print(f"TST graph nodes: {G_tst.number_of_nodes()}")
print(f"TST graph edges: {G_tst.number_of_edges()}")

# ------------------------------------------------------------
# REAL PAGERANK CONVERGENCE
# ------------------------------------------------------------

plt.figure(figsize=(10,5))

for d in [0.60, 0.70, 0.85, 0.95]:

    _, history = pagerank_with_history(
        G_tst,
        alpha=d,
        max_iter=80,
        tol=1e-15
    )

    plt.plot(
        range(1, len(history)+1),
        history,
        marker='o',
        markersize=3,
        linewidth=2,
        label=f"damping={d}"
    )

# ------------------------------------------------------------
# VISUALIZATION
# ------------------------------------------------------------

plt.yscale("log")

plt.title(
    "Measured PageRank Convergence on TST Semantic Graph",
    fontsize=13,
    fontweight='bold'
)

plt.xlabel("Iteration")
plt.ylabel("L1 Error (log scale)")

plt.grid(True, alpha=0.3)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
# CELL 15 - GraphRAG SEMANTIC LOGIC NETWORK: Visualizing Top 40 High-Contradiction Claims
print("[SYSTEM STARTED] TrumorGPT Graph Architecture Engine is active.")

G = nx.Graph()
top_critical = df_plot.nlargest(40,'con_true').index
for idx in top_critical:
    G.add_node(df_test.iloc[idx]['claim_id'], label=df_test.iloc[idx]['label'])
X_norm = normalize(X_test)
for i in range(len(top_critical)):
    for j in range(i+1, len(top_critical)):
        if np.dot(X_norm[top_critical[i]], X_norm[top_critical[j]]) > 0.98:
            G.add_edge(df_test.iloc[top_critical[i]]['claim_id'],
                       df_test.iloc[top_critical[j]]['claim_id'])
plt.figure(figsize=(12,8))
pos = nx.spring_layout(G, k=0.5, seed=42)
node_colors = ["mediumseagreen" if G.nodes[n]['label']==1 else "indianred" for n in G.nodes]
nx.draw(G, pos, with_labels=True, node_color=node_colors, node_size=800,
        font_size=8, font_weight="bold", edge_color="gray", alpha=0.8)
true_patch  = mpatches.Patch(color='mediumseagreen', label='True Claim')
false_patch = mpatches.Patch(color='indianred',      label='False Claim')
plt.legend(handles=[true_patch,false_patch], loc='upper right', frameon=True, shadow=True)
plt.title("Semantic Logic Network (Top 40 High-Contradiction Nodes)", fontweight='bold')
plt.show()

In [ ]:
# CELL 16 — METRIC PLOTS (Imbalanced split)
# FIX: Confusion matrix matches the metrics — no fake "undetermined" row.
# If we want abstain logic, it's reported separately as selective accuracy.
print("\n[STEP 12] Visualizing PUBHEALTH IMBALANCED results...")

m = results_imbalanced
y_pred  = m['y_pred']
y_proba = m['y_proba']

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

# Honest 2x2 confusion matrix (no invented class)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', linewidths=1, linecolor='black',
            xticklabels=["Pred False", "Pred True"],
            yticklabels=["Actual False", "Actual True"], ax=axes[0])
axes[0].set_title("Confusion Matrix\n(PUBHEALTH Imbalanced)", fontweight='bold')

# Metrics bar chart — includes macro-F1 and minority-recall
labels = ['Accuracy', 'F1\n(macro)', 'F1\n(weighted)', 'Recall\n(False)', 'AUC']
vals   = [m['accuracy'], m['f1_macro'], m['f1_weighted'],
          m['recall_minority'], m['auc']]
bars = axes[1].bar(labels, vals,
                    color=['#2ca02c', '#d62728', '#1f77b4', '#ff7f0e', '#9467bd'],
                    edgecolor='black')
axes[1].set_ylim(0, 1.1)
axes[1].set_title("PUBHEALTH Imbalanced — Metrics", fontweight='bold')
for b in bars:
    axes[1].text(b.get_x() + b.get_width()/2, b.get_height() + 0.02,
                 f"{b.get_height():.3f}", ha='center', fontweight='bold', fontsize=9)

# ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[2].plot(fpr, tpr, color='#d62728', lw=2.5, label=f'PUBHEALTH (AUC={m["auc"]:.3f})')
axes[2].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[2].fill_between(fpr, tpr, alpha=0.1, color='#d62728')
axes[2].set_xlim([0, 1]); axes[2].set_ylim([0, 1.05])
axes[2].set_title("ROC — PUBHEALTH Imbalanced", fontweight='bold')
axes[2].legend(loc="lower right")
plt.tight_layout(); plt.savefig("fig_pubhealth_metrics.png", dpi=200); plt.show()

# OPTIONAL: report selective accuracy with abstain rule (consistent reporting)
abstain_mask = np.maximum(X_test[:, 0], X_test[:, 3]) < 0.30
n_abstain = int(abstain_mask.sum())
if n_abstain > 0:
    coverage = 1 - n_abstain / len(y_test)
    sel_acc  = accuracy_score(y_test[~abstain_mask], y_pred[~abstain_mask])
    print(f"  Abstain rule (max KB cosine < 0.30):")
    print(f"    Abstained on {n_abstain} of {len(y_test)} claims (coverage={coverage:.1%})")
    print(f"    Selective accuracy on covered claims: {sel_acc:.3f}")


In [ ]:
# CELL 17 — METRIC PLOTS (Balanced split)
print("\n[STEP 13] Visualizing PUBHEALTH BALANCED results...")

m_bal = results_balanced
y_bal_pred  = m_bal['y_pred']
y_bal_proba = m_bal['y_proba']

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

cm_bal = confusion_matrix(y_bal_test, y_bal_pred, labels=[0, 1])
sns.heatmap(cm_bal, annot=True, fmt='d', cmap='Greens', linewidths=1, linecolor='black',
            xticklabels=["Pred False", "Pred True"],
            yticklabels=["Actual False", "Actual True"], ax=axes[0])
axes[0].set_title("Confusion Matrix\n(PUBHEALTH Balanced)", fontweight='bold')

labels = ['Accuracy', 'F1\n(macro)', 'F1\n(weighted)', 'Recall\n(False)', 'AUC']
vals   = [m_bal['accuracy'], m_bal['f1_macro'], m_bal['f1_weighted'],
          m_bal['recall_minority'], m_bal['auc']]
bars = axes[1].bar(labels, vals,
                    color=['#2ca02c', '#d62728', '#1f77b4', '#ff7f0e', '#9467bd'],
                    edgecolor='black')
axes[1].set_ylim(0, 1.1)
axes[1].set_title("PUBHEALTH Balanced — Metrics", fontweight='bold')
for b in bars:
    axes[1].text(b.get_x() + b.get_width()/2, b.get_height() + 0.02,
                 f"{b.get_height():.3f}", ha='center', fontweight='bold', fontsize=9)

fpr_b, tpr_b, _ = roc_curve(y_bal_test, y_bal_proba)
axes[2].plot(fpr_b, tpr_b, color='#2ca02c', lw=2.5, label=f'Balanced (AUC={m_bal["auc"]:.3f})')
axes[2].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[2].fill_between(fpr_b, tpr_b, alpha=0.1, color='#2ca02c')
axes[2].set_xlim([0, 1]); axes[2].set_ylim([0, 1.05])
axes[2].set_title("ROC — PUBHEALTH Balanced", fontweight='bold')
axes[2].legend(loc='lower right')
plt.tight_layout(); plt.savefig("fig_balanced_metrics.png", dpi=200); plt.show()


In [ ]:
# CELL 18 — IMBALANCE ABLATION RESULTS (honest framing)
# REMOVED: comparison to GPT-3.5 / LLaMA / Claude / Gemini paper numbers.
# Those were on different test splits — not directly comparable.
# This table reports our two settings only, with FULL metrics.
print("\n[STEP 14] Imbalance Ablation: Imbalanced vs Balanced PUBHEALTH...")

comparison_df = pd.DataFrame({
    "Setting":          ["PUBHEALTH Imbalanced (2:1)", "PUBHEALTH Balanced (1:1)"],
    "Train size":       [len(y_train), len(y_bal_train)],
    "Test size":        [len(y_test), len(y_bal_test)],
    "Accuracy":         [results_imbalanced['accuracy'], results_balanced['accuracy']],
    "F1 (macro)":       [results_imbalanced['f1_macro'], results_balanced['f1_macro']],
    "F1 (weighted)":    [results_imbalanced['f1_weighted'], results_balanced['f1_weighted']],
    "Recall (False)":   [results_imbalanced['recall_minority'], results_balanced['recall_minority']],
    "Recall (True)":    [results_imbalanced['recall_majority'], results_balanced['recall_majority']],
    "AUC-ROC":          [results_imbalanced['auc'], results_balanced['auc']],
})

display(
    comparison_df.style
    .set_caption("TABLE 3 — Same Pipeline, Different Class Balance (PUBHEALTH)")
    .format({c: "{:.3f}" for c in
             ["Accuracy", "F1 (macro)", "F1 (weighted)",
              "Recall (False)", "Recall (True)", "AUC-ROC"]})
    .set_table_styles(styles).hide(axis="index")
)

# Side-by-side bar comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metric_labels = ['Accuracy', 'F1\n(macro)', 'F1\n(weighted)', 'Recall\n(False)', 'AUC']
imb_vals = [results_imbalanced['accuracy'], results_imbalanced['f1_macro'],
            results_imbalanced['f1_weighted'], results_imbalanced['recall_minority'],
            results_imbalanced['auc']]
bal_vals = [results_balanced['accuracy'], results_balanced['f1_macro'],
            results_balanced['f1_weighted'], results_balanced['recall_minority'],
            results_balanced['auc']]

x = np.arange(len(metric_labels))
w = 0.35
axes[0].bar(x - w/2, imb_vals, w, label='Imbalanced (2:1)', color='#ff7f0e', edgecolor='black')
axes[0].bar(x + w/2, bal_vals, w, label='Balanced (1:1)',  color='#2ca02c', edgecolor='black')
axes[0].set_xticks(x); axes[0].set_xticklabels(metric_labels)
axes[0].set_ylim(0, 1.1)
axes[0].set_title("Imbalance Ablation — Metric Comparison", fontweight='bold')
axes[0].legend()
for i, (a, b) in enumerate(zip(imb_vals, bal_vals)):
    axes[0].text(i - w/2, a + 0.02, f"{a:.3f}", ha='center', fontsize=8, fontweight='bold')
    axes[0].text(i + w/2, b + 0.02, f"{b:.3f}", ha='center', fontsize=8, fontweight='bold')

# ROC overlay
fpr_imb, tpr_imb, _ = roc_curve(y_test, results_imbalanced['y_proba'])
fpr_bal, tpr_bal, _ = roc_curve(y_bal_test, results_balanced['y_proba'])
axes[1].plot(fpr_imb, tpr_imb, color='#ff7f0e', lw=2.5,
             label=f'Imbalanced (AUC={results_imbalanced["auc"]:.3f})')
axes[1].plot(fpr_bal, tpr_bal, color='#2ca02c', lw=2.5,
             label=f'Balanced (AUC={results_balanced["auc"]:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random')
axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1.05])
axes[1].set_xlabel('False Positive Rate', fontweight='bold')
axes[1].set_ylabel('True Positive Rate', fontweight='bold')
axes[1].set_title("ROC — Imbalanced vs Balanced", fontweight='bold')
axes[1].legend(loc='lower right')
plt.tight_layout(); plt.savefig("fig_ablation_comparison.png", dpi=200); plt.show()

print("\n[INTERPRETATION]")
print(f"  Imbalanced macro-F1: {results_imbalanced['f1_macro']:.3f} | "
      f"Balanced macro-F1: {results_balanced['f1_macro']:.3f}")
print(f"  Recall on minority (False) class:")
print(f"    Imbalanced: {results_imbalanced['recall_minority']:.3f}")
print(f"    Balanced  : {results_balanced['recall_minority']:.3f}")
print()
print("  Honest reading: SVM-SMOTE+Tomek improves minority-class recall on the")
print("  imbalanced split, but the natural class skew at test time still constrains")
print("  overall macro-F1. Balanced sampling provides a cleaner upper bound.")
print()
print("  NOTE: We do not compare to LLM baselines (GPT-3.5, LLaMA, Claude, Gemini)")
print("  in this table because those numbers come from different papers using")
print("  different train/test splits and label mappings — the comparison is not")
print("  apples-to-apples without re-running them on this exact split.")


In [ ]:
# CELL 19 — CONFIDENCE PLOT + IN-DISTRIBUTION QUALITATIVE PROBE
# FIX: System test now uses claims drawn from the held-out test set
# (real ground truth available) instead of OOD COVID/5G prompts.
print("\n[STEP 15] Confidence Score vs Prediction Plot...")
plt.figure(figsize=(10, 5))
pred_labels = ["True" if p == 1 else "False" for p in results_imbalanced['y_pred']]
sns.stripplot(x=results_imbalanced['y_proba'], y=pred_labels, hue=y_test,
              palette={1: "mediumseagreen", 0: "indianred"},
              jitter=0.08, size=8, alpha=0.7, orient="h",
              edgecolor="black", linewidth=0.5)
plt.axvline(results_imbalanced['threshold'], color='gray', linestyle='--', linewidth=1.5,
            label=f'Decision threshold ({results_imbalanced["threshold"]:.3f})')
plt.title("Confidence Score vs Model Prediction (PUBHEALTH Imbalanced)", fontweight='bold')
plt.xlabel("P(True Fact)")
plt.ylabel("Prediction")
plt.legend(title="Ground Truth", labels=["False (0)", "True (1)", "Threshold"])
plt.tight_layout(); plt.show()

print("\n" + "=" * 80)
print("[STEP 15] QUALITATIVE PROBE — Sample test claims with model verdict")
print("=" * 80)
print("Note: probe uses in-distribution claims from the held-out test set.")
print("Out-of-distribution probes (e.g. COVID-specific prompts not in training)")
print("would not be reliable — model is only validated on PUBHEALTH-style claims.\n")

# Sample 4 test claims (2 True, 2 False) for qualitative inspection
sample_idx = []
for label in [1, 0]:
    sample_idx.extend(df_test[df_test['label'] == label].sample(2, random_state=7).index.tolist())

X_test_scaled_imb = results_imbalanced['scaler'].transform(X_test)

for i in sample_idx:
    claim     = df_test.iloc[i]['claim_text']
    true_lbl  = int(df_test.iloc[i]['label'])
    pred_lbl  = int(results_imbalanced['y_pred'][i])
    prob_true = float(results_imbalanced['y_proba'][i])
    cos_t     = float(X_test[i, 0])
    cos_f     = float(X_test[i, 3])
    kb_diff   = cos_t - cos_f
    max_cos   = max(cos_t, cos_f)

    print(f"\nClaim       : {claim[:100]}...")
    print(f"Ground truth: {'TRUE' if true_lbl == 1 else 'FALSE'}")
    print(f"Cos True / False  : {cos_t:.3f} / {cos_f:.3f}    (KB_Diff={kb_diff:+.3f})")
    if max_cos < 0.50:
        print(f"Verdict     : ABSTAIN (max KB cosine {max_cos:.3f} < 0.50)")
    else:
        verdict = 'TRUE' if pred_lbl == 1 else 'FALSE'
        conf    = prob_true if pred_lbl == 1 else 1 - prob_true
        correct = "✓" if pred_lbl == true_lbl else "✗"
        print(f"Verdict     : {verdict} (conf {conf:.1%}) {correct}")


In [ ]:
# CELL 20 — EXPLAINABLE AI (FEATURE IMPORTANCE)
print("\n[IMPACT REPORT] Feature Importance Chart...")

hybrid_model = results_imbalanced['model']

importances = np.zeros(len(FEAT_NAMES))
for bag_clf in hybrid_model.estimators_:
    bag_imp = np.zeros(len(FEAT_NAMES))
    for weight, tree in zip(bag_clf.estimator_weights_, bag_clf.estimators_):
        bag_imp += weight * tree.feature_importances_
    if bag_clf.estimator_weights_.sum() > 0:
        bag_imp /= bag_clf.estimator_weights_.sum()
    importances += bag_imp
importances /= len(hybrid_model.estimators_)

indices         = np.argsort(importances)[::-1]
sorted_features = [FEAT_NAMES[i] for i in indices]
sorted_imp      = importances[indices]

plt.figure(figsize=(11, 6))
sns.barplot(x=sorted_imp, y=sorted_features, palette='mako')
plt.title('Feature Importance (12-D Hybrid Features, PUBHEALTH Imbalanced)',
          fontweight='bold', fontsize=14)
plt.xlabel('Relative Importance', fontsize=11)
plt.ylabel('Feature', fontsize=11)
plt.tight_layout()
plt.savefig("fig_feature_importance.png", dpi=200); plt.show()

# Print sorted importance
print("\n  Feature importance (sorted):")
for f, v in zip(sorted_features, sorted_imp):
    print(f"    {f:15s} {v:.4f}")

# LDA + PCA analysis
sc = results_imbalanced['scaler']
lda = LinearDiscriminantAnalysis()
lda.fit(sc.transform(X_train), y_train)
pca = PCA()
pca.fit(sc.transform(X_train))
print(f"\n  LDA train-set separability     : {lda.score(sc.transform(X_train), y_train):.4f}")
print(f"  PCA top 2 components variance  : {pca.explained_variance_ratio_[:2].sum():.4f}")
print(f"  PCA top 3 components variance  : {pca.explained_variance_ratio_[:3].sum():.4f}")


In [ ]:
# CELL 21 — 3D PCA FEATURE SPACE
print("\n[STEP 16] 3D PCA Feature Space Visualization...")
from mpl_toolkits.mplot3d import Axes3D

pca3 = PCA(n_components=3)
X_test_3d = pca3.fit_transform(results_imbalanced['scaler'].transform(X_test))

fig = plt.figure(figsize=(13, 8))
ax  = fig.add_subplot(111, projection='3d')
ax.scatter(X_test_3d[y_test == 1, 0], X_test_3d[y_test == 1, 1], X_test_3d[y_test == 1, 2],
           c='mediumseagreen', alpha=0.5, s=40, label='True Fact')
ax.scatter(X_test_3d[y_test == 0, 0], X_test_3d[y_test == 0, 1], X_test_3d[y_test == 0, 2],
           c='indianred', alpha=0.8, s=55, edgecolors='black', label='Fake News')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_zlabel('PC3')
ax.set_title(f"3D PCA Feature Space (PUBHEALTH Imbalanced Test)\n"
             f"Top-3 explained variance: {pca3.explained_variance_ratio_.sum():.1%}",
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig("fig_pca3d.png", dpi=200); plt.show()


In [ ]:
# CELL 22 — INFERENCE EFFICIENCY (honest framing)
# FIX: Removed the fabricated CO2 constant and the unsupported
# "99.9% less than GPT-4" claim. We report wall-clock latency only.
print("\n" + "=" * 85)
print("[INFERENCE EFFICIENCY REPORT]")
print("=" * 85)

X_test_scaled_imb = results_imbalanced['scaler'].transform(X_test)
n_warm = 10
_ = results_imbalanced['model'].predict(X_test_scaled_imb[:n_warm])  # warmup

n_bench = min(100, len(X_test_scaled_imb))
start = time.perf_counter()
_ = results_imbalanced['model'].predict(X_test_scaled_imb[:n_bench])
end = time.perf_counter()

total_ms     = (end - start) * 1000
per_query_ms = total_ms / n_bench

print(f"  Test batch size       : {n_bench} claims")
print(f"  Total inference time  : {total_ms:.2f} ms")
print(f"  Avg per-query latency : {per_query_ms:.3f} ms / claim")
print(f"  Throughput            : {1000/per_query_ms:.0f} claims/sec (single CPU thread)")
print("-" * 85)
print("  Note: This measures ONLY the downstream classifier latency.")
print("  The full pipeline (SBERT encoding + DeBERTa NLI + PageRank lookup)")
print("  dominates wall-clock cost in production. End-to-end latency is")
print("  ~0.5-2s per claim on a T4 GPU, dominated by NLI inference.")
print("=" * 85)


In [ ]:
# CELL 23 — ACADEMIC NOTE FOR DEFENSE
print("\n--- ACADEMIC NOTE FOR DEFENSE ---\n")
print("CONTRIBUTIONS")
print("  1. Dual-KB retrieval: separate True/False knowledge bases yield KB_Diff,")
print("     a discriminative signal not present in single-KB GraphRAG systems.")
print("  2. SHKG + PageRank: entity co-occurrence graph injected as graph_score feature.")
print("     (Built train-only to prevent test-set leakage.)")
print("  3. SVM-SMOTE + TomekLinks: boundary-aware oversampling synthesizes minority")
print("     samples at the decision frontier, then Tomek removes overlap.")
print("  4. Linguistic priors: hedge_score, sens_score, num_score, question_flag.")
print("  5. Imbalance ablation: same pipeline on imbalanced (2:1) vs balanced (1:1)")
print("     samplings of PUBHEALTH isolates the imbalance variable.")
print("")
print("RESULTS (honest)")
print(f"  PUBHEALTH Imbalanced: acc={results_imbalanced['accuracy']:.3f}, "
      f"macro-F1={results_imbalanced['f1_macro']:.3f}, "
      f"recall(False)={results_imbalanced['recall_minority']:.3f}")
print(f"  PUBHEALTH Balanced  : acc={results_balanced['accuracy']:.3f}, "
      f"macro-F1={results_balanced['f1_macro']:.3f}, "
      f"recall(False)={results_balanced['recall_minority']:.3f}")
print("")
print("KNOWN LIMITATIONS (we acknowledge upfront)")
print("  - en_core_web_sm has no DISEASE entity type; biomedical NER (scispaCy)")
print("    would likely improve graph_score signal quality.")
print("  - Single dataset (PUBHEALTH); cross-dataset generalization not tested.")
print("    LIAR or FEVER would require domain-specific KB rebuilding.")
print("  - No comparison to LLM baselines on the same test split — paper-reported")
print("    GPT/LLaMA/Claude numbers are not directly comparable.")
print("  - Test set is small (180/120 claims); confidence intervals would be wide.")


In [ ]:
# CELL 24 — FULL PAPER METHOD: TST + Topic-Enhanced Sentence Centrality
#
# Full implementation of Article Sections III-B and III-C:
#   1. Topic-Enhanced Sentence Centrality (LDA + BERT, eta=0.7)
#   2. Topic-Specific TextRank (TST) — health-weighted PageRank (Eq.1-4)
#   3. TST-guided triple extraction -> knowledge graph
#   4. Jaccard similarity (Eq.5) -> prediction
#
# Combinations:
# A) PUBHEALTH Balanced (1:1) — no SMOTE (same dataset as my novelty balanced)
# B) PUBHEALTH Imbalanced (2:1) — SMOTE+TomekLinks (same pipeline as my novelty imbalanced)
#
# Novelty method remains untouched.

print('=' * 75)
print('[CELL 24] Full Paper Method: TST + Topic-Enhanced Centrality')
print('=' * 75)

import numpy as np
import networkx as nx
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             roc_auc_score, classification_report)
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ─── STEP A: LDA MODEL ──────────────────────────────────────────────────────
# Article Section III-B: LDA training with health domain corpus
print('\n[A] Training LDA model on PUBHEALTH training corpus...')

K_TOPICS  = 10
HEALTH_KW = {'health','medical','medicine','disease','virus','vaccine',
             'treatment','patient','hospital','doctor','drug','study',
             'research','clinical','risk','infection','pandemic','covid',
             'cancer','symptom','therapy','fda','cdc','who'}

all_train_texts = pd.concat([df_train, df_bal_train])['claim_text'].tolist()

lda_vectorizer = CountVectorizer(max_features=3000, stop_words='english',
                                 min_df=2, max_df=0.95)
X_lda_corpus   = lda_vectorizer.fit_transform(all_train_texts)

lda_model = LatentDirichletAllocation(
    n_components=K_TOPICS, random_state=42,
    max_iter=20, learning_method='batch')
lda_model.fit(X_lda_corpus)

vocab = lda_vectorizer.get_feature_names_out()
health_topics_tst = set()
for k in range(K_TOPICS):
    top_words = set(vocab[i] for i in lda_model.components_[k].argsort()[-15:])
    if top_words & HEALTH_KW:
        health_topics_tst.add(k)
print(f'  LDA: {K_TOPICS} topics | Health topics: {sorted(health_topics_tst)}')


def get_lda_vec(texts):
    X = lda_vectorizer.transform(texts)
    return lda_model.transform(X)  # (n, K)


# ─── STEP B: TOPIC-ENHANCED SENTENCE CENTRALITY ─────────────────────────────
# Makale Eq: vs = [eta*e_s_norm ; (1-eta)*t_s_norm],  eta=0.7
ETA = 0.7  # Article: 'we set eta=0.7'


def topic_enhanced_centrality(sentences, top_k=5):
    if len(sentences) == 0:
        return []
    if len(sentences) == 1:
        return [0]
    bert_embs = normalize(
        bert_model.encode(sentences, show_progress_bar=False), norm='l2')
    lda_vecs  = normalize(get_lda_vec(sentences), norm='l2')
    combined  = np.concatenate([ETA * bert_embs, (1 - ETA) * lda_vecs], axis=1)
    sim       = combined @ combined.T
    np.fill_diagonal(sim, 0)
    sim = np.clip(sim, 0, None)
    G   = nx.from_numpy_array(sim)
    try:
        pr = nx.pagerank(G, weight='weight', alpha=0.85, max_iter=500, tol=1e-6)
    except nx.PowerIterationFailedConvergence:
        pr = {i: 1/len(sentences) for i in range(len(sentences))}
    ranked = sorted(pr.keys(), key=lambda x: pr[x], reverse=True)
    return ranked[:min(top_k, len(ranked))]


# ─── STEP C: TOPIC-SPECIFIC TEXTRANK (TST) ──────────────────────────────────
# Article Section III-C, Eq.1-4
ALPHA_TST = 1.5  # Article: 'giving health-related topics 50% more influence'
D_TST     = 0.85 # Article: 'default setting in PageRank'


def topic_specific_textrank(sentences, top_k=8):
    if len(sentences) == 0:
        return []
    if len(sentences) == 1:
        return [0]
    n        = len(sentences)
    lda_vecs = get_lda_vec(sentences)  # (n, K)
    # R(vi): topic relevance score (Article formula)
    beta  = np.array([ALPHA_TST if k in health_topics_tst else 1.0
                      for k in range(K_TOPICS)])
    raw_R = lda_vecs @ beta
    denom = raw_R.sum() if raw_R.sum() > 0 else 1.0
    R     = raw_R / denom  # normalize: sum R(vi)=1
    # BERT cosine base weights
    bert_embs = normalize(
        bert_model.encode(sentences, show_progress_bar=False), norm='l2')
    W_base = bert_embs @ bert_embs.T
    np.fill_diagonal(W_base, 0)
    W_base = np.clip(W_base, 0, None)
    # Adjusted weight: w'_{j,i} = (R(vi)+R(vj))/2 * w_{j,i}  (Article)
    R_col  = R.reshape(1, -1)
    R_row  = R.reshape(-1, 1)
    W_adj  = (R_row + R_col) / 2 * W_base
    # Column-stochastic P (Makale Eq.2)
    col_sums = W_adj.sum(axis=0)
    col_sums[col_sums == 0] = 1.0
    P_base   = W_adj / col_sums
    # Transition matrix P' = d*P + (1-d)*E  (Article Eq.3)
    E       = np.outer(R, np.ones(n))
    P_prime = D_TST * P_base + (1 - D_TST) * E
    # Power iteration (Article Eq.4), tol=1e-6, max_iter=500
    tst_vec = np.ones(n) / n
    for _ in range(500):
        tst_new = P_prime @ tst_vec
        if np.linalg.norm(tst_new - tst_vec, ord=1) < 1e-6:
            break
        tst_vec = tst_new
    ranked = np.argsort(tst_vec)[::-1]
    return list(ranked[:min(top_k, n)])


# ─── STEP D: TST-GUIDED TRIPLE EXTRACTION ───────────────────────────────────
def extract_triples_tst(text, top_k_sent=5, top_k_kw=8):
    doc       = nlp(text)
    sentences = [s.text.strip() for s in doc.sents if len(s.text.strip()) > 10]
    if not sentences:
        sentences = [text]
    central_idx = topic_enhanced_centrality(sentences, top_k=top_k_sent)
    tst_idx     = topic_specific_textrank(sentences, top_k=top_k_kw)
    selected    = list(set(central_idx) | set(tst_idx))
    selected_sents = [sentences[i] for i in selected if i < len(sentences)]
    triples = set()
    for sent in selected_sents:
        sdoc = nlp(sent)
        ents = [e.text.lower() for e in sdoc.ents]
        for i in range(len(ents)):
            for j in range(i+1, len(ents)):
                triples.add((ents[i], 'co-occurs', ents[j]))
        for chunk in sdoc.noun_chunks:
            head = chunk.root.head
            if head.dep_ in ('nsubj','nsubjpass','dobj','pobj','attr'):
                subj = chunk.text.lower()
                rel  = head.lemma_.lower()
                obj  = (head.head.text.lower()
                        if head.head != head else 'unknown')
                triples.add((subj, rel, obj))
    return triples


def jac_sim(tx, ti):
    if not tx and not ti:
        return 0.0
    inter = len(tx & ti)
    union = len(tx | ti)
    return inter / union if union > 0 else 0.0


# ─── STEP E: FULL PAPER METHOD EVALUATION ───────────────────────────────────
# apply_smote=True -> Apply SMOTE+TomekLinks for imbalanced dataset
# (same pipeline as my novelty imbalanced)
# apply_smote=False -> for balanced dataset (same dataset as my novelty balanced)
def full_paper_evaluate(df_train_split, df_test_split,
                         dataset_name, apply_smote=False):
    smote_tag = ' + SMOTE+TomekLinks' if apply_smote else ' (no resampling)'
    print(f'\n{"="*65}')
    print(f'FULL PAPER METHOD{smote_tag} -- {dataset_name}')
    print(f'{"="*65}')
    print(f'  Train: {(df_train_split["label"]==1).sum()} True | '
          f'{(df_train_split["label"]==0).sum()} False')
    print(f'  Test : {(df_test_split["label"]==1).sum()} True | '
          f'{(df_test_split["label"]==0).sum()} False')

    # KB triple'larini TST+Centrality ile olustur (train only, no leakage)
    print('  Building KB triples with TST+Centrality (train only)...')
    kb_true  = [extract_triples_tst(t)
                for t in tqdm(df_train_split[df_train_split['label']==1]
                              ['claim_text'].tolist(), desc='  True KB')]
    kb_false = [extract_triples_tst(t)
                for t in tqdm(df_train_split[df_train_split['label']==0]
                              ['claim_text'].tolist(), desc='  False KB')]

    # Test set Jaccard scores
    print('  Computing Jaccard scores on test set...')
    scores, y_true = [], []
    for _, row in tqdm(df_test_split.iterrows(), total=len(df_test_split)):
        q   = extract_triples_tst(row['claim_text'])
        bt  = max((jac_sim(q, t) for t in kb_true),  default=0.0)
        bf  = max((jac_sim(q, t) for t in kb_false), default=0.0)
        scores.append(bt - bf)
        y_true.append(int(row['label']))

    # Train scores (for threshold tuning)
    print('  Computing train scores for threshold tuning...')
    tr_sc, tr_lbl = [], []
    for _, row in tqdm(df_train_split.iterrows(), total=len(df_train_split)):
        q   = extract_triples_tst(row['claim_text'])
        bt  = max((jac_sim(q, t) for t in kb_true),  default=0.0)
        bf  = max((jac_sim(q, t) for t in kb_false), default=0.0)
        tr_sc.append(bt - bf)
        tr_lbl.append(int(row['label']))

    scores  = np.array(scores)
    y_true  = np.array(y_true)
    tr_sc   = np.array(tr_sc)
    tr_lbl  = np.array(tr_lbl)

    # SMOTE+TomekLinks: ONLY for imbalanced combinations
    # Same pipeline as my novelty method -> fair comparison
    if apply_smote:
        print('  Applying SVM-SMOTE + TomekLinks on Jaccard score feature...')
        sc_2d = tr_sc.reshape(-1, 1)
        smote_obj = SVMSMOTE(
            sampling_strategy='auto', random_state=42,
            k_neighbors=min(5, (tr_lbl==0).sum()-1),
            m_neighbors=10,
            svm_estimator=SVC(kernel='rbf', gamma='scale', random_state=42))
        sc_res, lab_res = smote_obj.fit_resample(sc_2d, tr_lbl)
        tomek_obj = TomekLinks(sampling_strategy='majority')
        sc_res, lab_res = tomek_obj.fit_resample(sc_res, lab_res)
        tr_sc_use  = sc_res.ravel()
        tr_lbl_use = lab_res
        print(f'  After SMOTE+Tomek: {(tr_lbl_use==1).sum()} True | '
              f'{(tr_lbl_use==0).sum()} False')
    else:
        tr_sc_use  = tr_sc
        tr_lbl_use = tr_lbl

    # Threshold: macro-F1 optimized from val slice
    idx_fit, idx_val = train_test_split(
        range(len(tr_lbl_use)), test_size=0.2,
        random_state=42, stratify=tr_lbl_use)
    val_sc  = tr_sc_use[list(idx_val)]
    val_lbl = tr_lbl_use[list(idx_val)]
    best_thr, best_f1 = 0.0, 0.0
    for thr in np.linspace(tr_sc_use.min(), tr_sc_use.max(), 300):
        preds  = (val_sc >= thr).astype(int)
        f1_val = f1_score(val_lbl, preds, average='macro', zero_division=0)
        if f1_val > best_f1:
            best_f1, best_thr = f1_val, float(thr)

    y_pred  = (scores >= best_thr).astype(int)
    scale   = max(abs(scores).max(), 1e-6)
    y_proba = 1 / (1 + np.exp(-scores / scale * 4))

    metrics = {
        'dataset':         dataset_name,
        'accuracy':        accuracy_score(y_true, y_pred),
        'f1_macro':        f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted':     f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_minority': recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        'recall_majority': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'auc':             roc_auc_score(y_true, y_proba),
        'threshold':       best_thr,
        'y_pred':          y_pred,
        'y_proba':         y_proba,
        'y_true':          y_true,
    }

    print(f'\n  RESULTS -- {dataset_name}{smote_tag}')
    print(f'  Accuracy        : {metrics["accuracy"]:.3f}')
    print(f'  F1 (macro)      : {metrics["f1_macro"]:.3f}')
    print(f'  F1 (weighted)   : {metrics["f1_weighted"]:.3f}')
    print(f'  Recall (False)  : {metrics["recall_minority"]:.3f}')
    print(f'  AUC-ROC         : {metrics["auc"]:.3f}')
    print()
    print(classification_report(y_true, y_pred, zero_division=0,
                                target_names=['False','True']))
    return metrics


# ─── RUN ───────────────────────────────────────────────────────────────
print('\n>>> Comb. A: Full Paper Method | Balanced (1:1) | no SMOTE')
print('    (Same dataset as Novelty Balanced -> Fair comparison)')
results_full_paper_balanced = full_paper_evaluate(
    df_bal_train, df_bal_test,
    dataset_name='PUBHEALTH Balanced (1:1)',
    apply_smote=False)

print('\n>>> Comb. B: Full Paper Method | Imbalanced (2:1) | SMOTE+TomekLinks')
print('    (Same pipeline as Novelty Imbalanced -> Fair Comparison)')
results_full_paper_imbalanced = full_paper_evaluate(
    df_train, df_test,
    dataset_name='PUBHEALTH Imbalanced (2:1)',
    apply_smote=True)

print('\n[CELL 24 COMPLETE] Full paper method results ready.')


In [ ]:
# CELL 25 — FINAL COMPARISON TABLE
#
# | Satyr | Method | Dataset | Resampling |
# |----------|--------------|-----------------|------------------|
# | 1 | TrumorGPT (TST+Centrality+Jaccard) | Balanced (1:1) | None |
# | 2 | TrumorGPT (TST+Centrality+Jaccard) | Imbalanced (2:1)| SMOTE+TomekLinks |
# | 3 | Novelty (Dual-KB+SMOTE+Bagging) | Balanced (1:1) | None (1:1 anyway) |
# | 4 | Novelty (Dual-KB+SMOTE+Bagging) | Imbalanced (2:1)| SMOTE+TomekLinks |
# | ref | TrumorGPT GPT-4 (original article) | PolitiFact | - (different dataset)|
#
# Fair comparison: paper vs novelty on the same line SAME dataset + SAME resampling

print('\n' + '=' * 80)
print('[CELL 25] Comparison Table')
print('=' * 80)

from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def make_row(res, y_true_arr, method_name, dataset_label, resampling, notes=''):
    prec, rec, _, _ = precision_recall_fscore_support(
        y_true_arr, res['y_pred'], average='macro', zero_division=0)
    return {
        'Method':         method_name,
        'Dataset':        dataset_label,
        'Resampling':     resampling,
        'Accuracy':       round(res['accuracy'],        3),
        'Precision':      round(float(prec),            3),
        'Recall':         round(float(rec),             3),
        'F1 (weighted)':  round(res['f1_weighted'],     3),
        'F1 (macro)':     round(res['f1_macro'],        3),
        'Recall (False)': round(res['recall_minority'], 3),
        'AUC-ROC':        round(res['auc'],             3),
        'Notes':          notes,
    }


rows = [
    # --- Row 1: Paper | Balanced | no SMOTE ---
    make_row(
        results_full_paper_balanced,
        results_full_paper_balanced['y_true'],
        method_name   = 'TrumorGPT\n(TST+Centrality+Jaccard)',
        dataset_label = 'Balanced (1:1)',
        resampling    = 'No',

    ),
    # --- Row 2: Paper | Imbalanced | SMOTE+Tomek ---
    make_row(
        results_full_paper_imbalanced,
        results_full_paper_imbalanced['y_true'],
        method_name   = 'TrumorGPT\n(TST+Centrality+Jaccard+SMOTE+Bagging)',
        dataset_label = 'Imbalanced (2:1)',
        resampling    = 'SVM-SMOTE + TomekLinks',

    ),
    # --- Row 3: Novelty | Balanced | no SMOTE ---
    make_row(
        results_balanced,
        y_bal_test,
        method_name   = 'Novelty Method\n(Dual-KB)',
        dataset_label = 'Balanced (1:1)',
        resampling    = 'No',

    ),
    # --- Row 4: Novelty | Imbalanced | SMOTE+Tomek ---
    make_row(
        results_imbalanced,
        y_test,
        method_name   = 'Novelty Method\n(Dual-KB+SMOTE+Bagging)',
        dataset_label = 'Imbalanced (2:1)',
        resampling    = 'SVM-SMOTE + TomekLinks',

    ),
]

# Original article result (reference — different dataset)
ref_row = {
    'Method':         'TrumorGPT GPT-4 (paper original) ',
    'Dataset':        'PolitiFact 600 claims',
    'Resampling':     '-',
    'Accuracy':       0.885,
    'Precision':      0.914,
    'Recall':         0.850,
    'F1 (weighted)':  0.881,
    'F1 (macro)':     '-',
    'Recall (False)': '-',
    'AUC-ROC':        '-',
    'Notes':          'different dataset -- referans only',
}

final_df = pd.concat(
    [pd.DataFrame(rows), pd.DataFrame([ref_row])],
    ignore_index=True
)

numeric_cols = ['Accuracy','Precision','Recall',
                'F1 (weighted)','F1 (macro)','Recall (False)','AUC-ROC']


def fmt_cell(x):
    if isinstance(x, float):
        return f'{x:.3f}'
    return x


def row_color(col):
    colors = []
    for i, _ in col.items():
        method = str(final_df.loc[i, 'Method'])
        dset   = str(final_df.loc[i, 'Dataset'])
        if 'GPT-4' in method:
            colors.append('background-color: #f4cccc')
        elif 'Novelty' in method and '1:1' in dset:
            colors.append('background-color: #d9ead3')
        elif 'Novelty' in method:
            colors.append('background-color: #c6efce')
        elif 'TST' in method and '1:1' in dset:
            colors.append('background-color: #cfe2f3')
        else:
            colors.append('background-color: #bdd7ee')
    return colors


numeric_float_cols = [
    c for c in numeric_cols
    if final_df[c].apply(lambda x: isinstance(x, float)).all()
]

styles = [
    dict(selector='caption', props=[('font-size','13px'),('font-weight','bold'),
                                    ('text-align','left'),('margin-bottom','8px')]),
    dict(selector='th', props=[('text-align','center'),
                               ('background-color','#f4f4f4'),
                               ('font-size','14px')]),
    dict(selector='td', props=[('text-align','center'),
                               ('font-size','14px')]),
]

styled = (
    final_df.style
    .set_caption(
        'Comparison Table  '

    )
    .format({c: fmt_cell for c in numeric_cols})
    .apply(row_color, subset=['Method'])
    .highlight_max(subset=numeric_float_cols, color='#ffd966')
    .set_table_styles(styles)
    .hide(axis='index')
)
display(styled)

# ─── BAR CHART ──────────────────────────────────────────────────────────────
labels_short = [
    'Paper\nBalanced\n(no SMOTE)',
    'Paper\nImbalanced\n(+SMOTE)',
    'Novelty\nBalanced\n(no SMOTE)',
    'Novelty\nImbalanced\n(+SMOTE)',
]
accs  = [rows[i]['Accuracy']     for i in range(4)]
f1s   = [rows[i]['F1 (weighted)'] for i in range(4)]
aucs  = [rows[i]['AUC-ROC']       for i in range(4)]
bar_colors = ['#6baed6','#2171b5','#31a354','#006d2c']
x = np.arange(len(labels_short))
w = 0.55

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, vals, title, ref_val, ref_lbl in [
    (axes[0], accs, 'Accuracy',      0.885, 'Paper GPT-4 ref (0.885)'),
    (axes[1], f1s,  'F1 (weighted)', 0.881, 'Paper GPT-4 ref (0.881)'),
    (axes[2], aucs, 'AUC-ROC',       None,  None),
]:
    bars = ax.bar(x, vals, color=bar_colors, edgecolor='black', width=w)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_short, fontsize=8.5)
    ax.set_ylim(0.00, 1.05)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_ylabel(title)
    if ref_val:
        ax.axhline(ref_val, color='red', linestyle='--',
                   linewidth=1.5, label=ref_lbl)
        ax.legend(fontsize=8)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.013,
                f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')

p1 = mpatches.Patch(color='#6baed6', label='Paper (TST+Centrality) | Balanced | no SMOTE')
p2 = mpatches.Patch(color='#2171b5', label='Paper (TST+Centrality) | Imbalanced | +SMOTE')
p3 = mpatches.Patch(color='#31a354', label='Novelty | Balanced | no SMOTE')
p4 = mpatches.Patch(color='#006d2c', label='Novelty | Imbalanced | +SMOTE')
fig.legend(handles=[p1,p2,p3,p4], loc='lower center', ncol=2,
           fontsize=9, frameon=True, bbox_to_anchor=(0.5,-0.12))

plt.suptitle(
    'TrumorGPT vs Novelty Method | PUBHEALTH\n',
    fontweight='bold', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('fig_final_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nCOMPARISON PAIRS:')
print(f'  Row 1 vs Row 3 (Balanced, no SMOTE):')
print(f'    Paper  -> Acc:{rows[0]["Accuracy"]:.3f} | F1w:{rows[0]["F1 (weighted)"]:.3f} | AUC:{rows[0]["AUC-ROC"]:.3f}')
print(f'    Novelty-> Acc:{rows[2]["Accuracy"]:.3f} | F1w:{rows[2]["F1 (weighted)"]:.3f} | AUC:{rows[2]["AUC-ROC"]:.3f}')
print()
print(f'  Row 2 vs Row 4 (Imbalanced, +SMOTE+Tomek):')
print(f'    Paper  -> Acc:{rows[1]["Accuracy"]:.3f} | F1w:{rows[1]["F1 (weighted)"]:.3f} | AUC:{rows[1]["AUC-ROC"]:.3f}')
print(f'    Novelty-> Acc:{rows[3]["Accuracy"]:.3f} | F1w:{rows[3]["F1 (weighted)"]:.3f} | AUC:{rows[3]["AUC-ROC"]:.3f}')
